# Few-Shot Mammography Classification — MLO View

View-specific few-shot comparison of a compact CNN, VGG16, ResNet50, and CapsNet
on INbreast mammograms. Set `VIEW` and `N_PER_CLASS` in the **Configuration** cell
to reproduce any row of Table I in the paper.

**Paper:** *A View-Specific Few-Shot Comparison of Capsule Networks and Standard CNN Backbones on INbreast Mammography*  
**DOI:** 10.1109/ICIPCN67432.2026.11438702


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, GlobalAveragePooling2D,
    Flatten, Dense, Dropout, BatchNormalization
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam as KerasAdam
from keras.applications.vgg16 import VGG16
from tensorflow.keras.applications.resnet50 import ResNet50
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

## Configuration

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
# Set ROOT to your local preprocessed image folder (must contain CC/ and MLO/)
ROOT = "path/to/image/folder"

LABELS_CSV  = "path/to/INbreast.csv"
VIEW        = "MLO"       # "CC" or "MLO"
N_PER_CLASS = 10  # few-shot samples per class: 10, 20, or 31
N_FOLDS     = 5
SEED        = 0

# Training
IMG_SIZE    = (224, 224)
BATCH_SIZE  = 8
OUTDIR      = "./fold_models"
os.makedirs(OUTDIR, exist_ok=True)

# Fix seeds
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Data Loading & Splits

In [ ]:
df = pd.read_csv(LABELS_CSV, sep=";")
df.columns = [c.strip() for c in df.columns]

def map_birads_to_label(birads):
    return "normal" if str(birads).strip() in ["1", "2"] else "abnormal"

df["label"] = df["Bi-Rads"].apply(map_birads_to_label)

def build_path(row):
    fname = str(row["File Name"]).strip()
    view  = str(row["View"]).strip().upper()
    return os.path.join(ROOT, view, fname + ".jpeg").replace("\\", "/")

df["image_path"] = df.apply(build_path, axis=1)
df_view = df[df["View"] == VIEW].copy()

print(f"{VIEW} — normal: {(df_view['label']=='normal').sum()}, "
      f"abnormal: {(df_view['label']=='abnormal').sum()}")

In [ ]:
train_df, test_df = train_test_split(
    df_view, test_size=0.2, stratify=df_view["label"], random_state=SEED
)
train_df, val_df = train_test_split(
    train_df, test_size=0.2, stratify=train_df["label"], random_state=SEED
)
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

In [ ]:
def make_fewshot_splits(df, n_per_class, n_folds, seed):
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    folds = []
    for fold, (train_idx, val_idx) in enumerate(skf.split(df, df["label"]), start=1):
        fold_train = df.iloc[train_idx].groupby("label", group_keys=False).apply(
            lambda x: x.sample(n=min(n_per_class, len(x)), random_state=seed)
        )
        fold_val = df.iloc[val_idx]
        folds.append((fold, fold_train, fold_val))
    return folds

fewshot_folds = make_fewshot_splits(train_df, N_PER_CLASS, N_FOLDS, SEED)
for fold, tr, va in fewshot_folds:
    print(f"Fold {fold}: {len(tr)} train, {len(va)} val")

## Keras Augmentation Pipelines

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    rotation_range=15,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.05
)
val_datagen  = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

## Model Definitions

In [ ]:
def create_compact_cnn(input_shape=(224, 224, 3)):
    """Three-block compact CNN — non-pretrained baseline."""
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=input_shape, padding='same'),
        MaxPooling2D((2, 2)),
        BatchNormalization(),

        Conv2D(64, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),
        BatchNormalization(),

        Conv2D(128, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),
        BatchNormalization(),

        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=KerasAdam(learning_rate=5e-5),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

In [ ]:
def create_vgg16(input_shape=(224, 224, 3)):
    """Frozen VGG16 backbone with a dense classifier head."""
    backbone = VGG16(input_shape=input_shape, include_top=False, weights='imagenet')
    backbone.trainable = False
    model = Sequential([
        backbone,
        Flatten(),
        Dense(512, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=KerasAdam(learning_rate=5e-5),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

In [ ]:
def create_resnet50(input_shape=(224, 224, 3)):
    """Frozen ResNet50 backbone with a global average pooling head."""
    backbone = ResNet50(input_shape=input_shape, include_top=False, weights='imagenet')
    backbone.trainable = False
    model = Sequential([
        backbone,
        GlobalAveragePooling2D(),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=KerasAdam(learning_rate=5e-5),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

## Keras Training Loop

In [ ]:
def run_keras_experiment(model_fn, model_tag, fewshot_folds, test_df,
                         epochs=30, patience=6):
    """
    Train model_fn across all fewshot_folds, evaluate on test_df.
    Returns a DataFrame of per-fold results.
    """
    fold_results = []

    for fold_number, train_df_fold, val_df_fold in fewshot_folds:
        print(f"\n===== Fold {fold_number} ({len(train_df_fold)} train, "
              f"{len(val_df_fold)} val) =====")

        classes = np.unique(train_df_fold['label'])
        cw = compute_class_weight(
            class_weight='balanced', classes=classes, y=train_df_fold['label']
        )

        train_gen = train_datagen.flow_from_dataframe(
            train_df_fold.reset_index(drop=True),
            x_col="image_path", y_col="label",
            target_size=IMG_SIZE, color_mode="rgb",
            class_mode="binary", batch_size=BATCH_SIZE,
            shuffle=True, seed=SEED
        )
        val_gen = val_datagen.flow_from_dataframe(
            val_df_fold.reset_index(drop=True),
            x_col="image_path", y_col="label",
            target_size=IMG_SIZE, color_mode="rgb",
            class_mode="binary", batch_size=BATCH_SIZE,
            shuffle=False
        )
        test_gen = test_datagen.flow_from_dataframe(
            test_df.reset_index(drop=True),
            x_col="image_path", y_col="label",
            target_size=IMG_SIZE, color_mode="rgb",
            class_mode="binary", batch_size=BATCH_SIZE,
            shuffle=False
        )

        class_indices    = train_gen.class_indices
        class_weight_dict = {
            class_indices[lbl]: float(cw[np.where(classes == lbl)[0][0]])
            for lbl in class_indices
        }

        model      = model_fn()
        model_path = os.path.join(OUTDIR, f"{model_tag}_fold_{fold_number}.h5")

        callbacks = [
            ModelCheckpoint(model_path, monitor='val_accuracy',
                            save_best_only=True, mode='max', verbose=0),
            EarlyStopping(monitor='val_accuracy', patience=patience,
                          restore_best_weights=True, mode='max', verbose=1),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                              patience=3, verbose=1, min_lr=1e-6)
        ]

        model.fit(
            train_gen,
            steps_per_epoch=max(1, int(np.ceil(len(train_df_fold) / BATCH_SIZE))),
            validation_data=val_gen,
            validation_steps=max(1, int(np.ceil(len(val_df_fold) / BATCH_SIZE))),
            epochs=epochs,
            class_weight=class_weight_dict,
            callbacks=callbacks,
            verbose=1
        )

        try:
            model = tf.keras.models.load_model(model_path)
        except Exception as e:
            print(f"Warning: could not reload checkpoint: {e}")

        # Validation metrics
        val_gen_eval = val_datagen.flow_from_dataframe(
            val_df_fold.reset_index(drop=True),
            x_col="image_path", y_col="label",
            target_size=IMG_SIZE, color_mode="rgb",
            class_mode="binary", batch_size=BATCH_SIZE, shuffle=False
        )
        val_loss, val_acc = model.evaluate(val_gen_eval, verbose=0)
        val_preds         = model.predict(val_gen_eval, verbose=0).ravel()
        val_pred_classes  = (val_preds > 0.5).astype(int)

        # Test metrics
        test_loss, test_acc = model.evaluate(test_gen, verbose=0)
        test_preds          = model.predict(test_gen, verbose=0).ravel()
        test_pred_classes   = (test_preds > 0.5).astype(int)

        fold_results.append({
            "fold":           fold_number,
            "train_samples":  len(train_df_fold),
            "val_acc":        float(val_acc),
            "val_loss":       float(val_loss),
            "val_precision":  float(precision_score(val_gen_eval.labels, val_pred_classes)),
            "val_recall":     float(recall_score(val_gen_eval.labels, val_pred_classes)),
            "val_f1":         float(f1_score(val_gen_eval.labels, val_pred_classes)),
            "val_auc":        float(roc_auc_score(val_gen_eval.labels, val_preds)),
            "test_acc":       float(test_acc),
            "test_loss":      float(test_loss),
            "test_precision": float(precision_score(test_gen.labels, test_pred_classes)),
            "test_recall":    float(recall_score(test_gen.labels, test_pred_classes)),
            "test_f1":        float(f1_score(test_gen.labels, test_pred_classes)),
            "test_auc":       float(roc_auc_score(test_gen.labels, test_preds)),
        })
        print(f"Fold {fold_number} — val_acc: {val_acc:.4f}, test_acc: {test_acc:.4f}")

    return pd.DataFrame(fold_results).sort_values("fold").reset_index(drop=True)

## Run — Compact CNN

In [ ]:
results_cnn = run_keras_experiment(
    create_compact_cnn, f"cnn_{VIEW}_{N_PER_CLASS}shot",
    fewshot_folds, test_df
)
results_cnn.to_csv(f"{VIEW}_CNN_{N_PER_CLASS}.csv", index=False)
display(results_cnn)

## Run — VGG16

In [ ]:
results_vgg = run_keras_experiment(
    create_vgg16, f"vgg_{VIEW}_{N_PER_CLASS}shot",
    fewshot_folds, test_df
)
results_vgg.to_csv(f"{VIEW}_VGG_{N_PER_CLASS}.csv", index=False)
display(results_vgg)

## Run — ResNet50

In [ ]:
results_resnet = run_keras_experiment(
    create_resnet50, f"resnet_{VIEW}_{N_PER_CLASS}shot",
    fewshot_folds, test_df
)
results_resnet.to_csv(f"{VIEW}_ResNet_{N_PER_CLASS}.csv", index=False)
display(results_resnet)

## Capsule Network

In [ ]:
class MammoDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.transform = transform
        self.label_map = {"normal": 0, "abnormal": 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        image = torch.tensor(np.array(image, dtype=np.float32) / 255.0)
        image = image.permute(2, 0, 1)  # HWC -> CHW
        image = (image - 0.5) / 0.5    # match Normalize([0.5], [0.5])
        return image, torch.tensor(self.label_map[row["label"]], dtype=torch.long)

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
])

In [ ]:
class CapsuleLayer(nn.Module):
    def __init__(self, num_capsules, num_routes, in_channels, out_channels, num_iterations=3):
        super().__init__()
        self.num_routes    = num_routes
        self.num_capsules  = num_capsules
        self.num_iterations = num_iterations
        self.W = nn.Parameter(
            0.01 * torch.randn(num_routes, num_capsules, in_channels, out_channels)
        )

    def forward(self, x):
        x     = x[:, :, None, :].expand(-1, -1, self.num_capsules, -1)
        u_hat = torch.matmul(x.unsqueeze(3), self.W).squeeze(3)
        b_ij  = torch.zeros(x.size(0), self.num_routes, self.num_capsules, device=x.device)
        for i in range(self.num_iterations):
            c_ij = F.softmax(b_ij, dim=2)
            s_j  = (c_ij.unsqueeze(-1) * u_hat).sum(dim=1)
            v_j  = self.squash(s_j)
            if i < self.num_iterations - 1:
                b_ij = b_ij + (u_hat * v_j[:, None, :, :]).sum(-1)
        return v_j

    @staticmethod
    def squash(s, dim=-1):
        norm  = (s ** 2).sum(dim=dim, keepdim=True)
        scale = norm / (1 + norm)
        return scale * s / torch.sqrt(norm + 1e-8)


class CapsuleNet(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.conv1           = nn.Conv2d(3, 256, kernel_size=9, stride=1)
        self.primary_capsules = nn.Conv2d(256, 32 * 8, kernel_size=9, stride=2)
        self.capsules        = None
        self.num_classes     = num_classes

    def forward(self, x):
        x         = F.relu(self.conv1(x), inplace=True)
        x         = self.primary_capsules(x)
        B, C, H, W = x.size()
        x         = x.view(B, 32, 8, H, W).permute(0, 3, 4, 1, 2).contiguous()
        x         = x.view(B, H * W * 32, 8)
        if self.capsules is None:
            self.capsules = CapsuleLayer(
                num_capsules=self.num_classes,
                num_routes=H * W * 32,
                in_channels=8,
                out_channels=16
            ).to(x.device)
        return torch.norm(self.capsules(x), dim=-1)


class CapsuleLoss(nn.Module):
    def __init__(self, m_plus=0.9, m_minus=0.1, lambda_=0.5):
        super().__init__()
        self.m_plus  = m_plus
        self.m_minus = m_minus
        self.lambda_ = lambda_

    def forward(self, preds, labels):
        labels = F.one_hot(labels, num_classes=preds.size(1)).float()
        left   = F.relu(self.m_plus  - preds) ** 2
        right  = F.relu(preds - self.m_minus) ** 2
        return (labels * left + self.lambda_ * (1 - labels) * right).sum(dim=1).mean()

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = correct = total = 0
    for images, labels in tqdm(loader, leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct      += (outputs.argmax(dim=1) == labels).sum().item()
        total        += labels.size(0)
    return running_loss / total, correct / total


def evaluate_capsnet(model, loader, criterion):
    model.eval()
    running_loss = correct = total = 0
    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs        = model(images)
            loss           = criterion(outputs, labels)
            running_loss  += loss.item() * images.size(0)
            probs          = torch.softmax(outputs, dim=1)[:, 1]
            preds          = outputs.argmax(dim=1)
            correct       += (preds == labels).sum().item()
            total         += labels.size(0)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    avg_loss = running_loss / total
    accuracy = correct / total
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall    = recall_score(all_labels, all_preds, zero_division=0)
    f1        = f1_score(all_labels, all_preds, zero_division=0)
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        auc = float('nan')
    return avg_loss, accuracy, precision, recall, f1, auc

## Run — CapsNet

In [ ]:
CAPS_EPOCHS = 20
CAPS_LR     = 1e-3

fold_results_caps = []

for fold_number, train_df_fold, val_df_fold in fewshot_folds:
    print(f"\n===== Fold {fold_number} =====")

    train_loader = DataLoader(
        MammoDataset(train_df_fold, train_transform), batch_size=BATCH_SIZE, shuffle=True
    )
    val_loader = DataLoader(
        MammoDataset(val_df_fold, val_transform), batch_size=BATCH_SIZE, shuffle=False
    )
    test_loader = DataLoader(
        MammoDataset(test_df, val_transform), batch_size=BATCH_SIZE, shuffle=False
    )

    model     = CapsuleNet(num_classes=2).to(device)
    criterion = CapsuleLoss()
    optimizer = Adam(model.parameters(), lr=CAPS_LR)

    best_val_acc = 0
    best_metrics = {}
    ckpt_path    = os.path.join(OUTDIR, f"capsnet_{VIEW}_{N_PER_CLASS}shot_fold_{fold_number}.pt")

    for epoch in range(CAPS_EPOCHS):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc, val_prec, val_rec, val_f1, val_auc = evaluate_capsnet(
            model, val_loader, criterion
        )
        print(f"Epoch {epoch+1}/{CAPS_EPOCHS} — "
              f"train_loss: {train_loss:.4f}, train_acc: {train_acc:.4f} | "
              f"val_acc: {val_acc:.4f}, val_f1: {val_f1:.4f}, val_auc: {val_auc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_metrics = dict(val_loss=val_loss, val_acc=val_acc,
                                val_precision=val_prec, val_recall=val_rec,
                                val_f1=val_f1, val_auc=val_auc)
            torch.save(model.state_dict(), ckpt_path)

    model.load_state_dict(torch.load(ckpt_path))
    test_loss, test_acc, test_prec, test_rec, test_f1, test_auc = evaluate_capsnet(
        model, test_loader, criterion
    )
    print(f"Fold {fold_number} Test — acc: {test_acc:.4f}, f1: {test_f1:.4f}, auc: {test_auc:.4f}")

    fold_results_caps.append({
        "fold":           fold_number,
        "train_samples":  len(train_df_fold),
        **{f"val_{k}": float(v) for k, v in best_metrics.items()},
        "test_acc":       float(test_acc),
        "test_loss":      float(test_loss),
        "test_precision": float(test_prec),
        "test_recall":    float(test_rec),
        "test_f1":        float(test_f1),
        "test_auc":       float(test_auc),
    })

results_caps_df = pd.DataFrame(fold_results_caps).sort_values("fold").reset_index(drop=True)
results_caps_df.to_csv(f"{VIEW}_CapsNet_{N_PER_CLASS}.csv", index=False)
display(results_caps_df)